In [5]:
import pickle
import networkx as nx

# Paths to your files (update these if needed)
FULL_GRAPH_PATH = "../graph/aon_pruned_graph.gpickle"
SMALL_GRAPH_PATH = "../graph/aon_pruned_graph_freq100.gpickle"

print("Loading Full Graph...")
with open(FULL_GRAPH_PATH, "rb") as f:
    G_full = pickle.load(f)
print(f"Full Graph loaded: {G_full.number_of_nodes():,} nodes, {G_full.number_of_edges():,} edges")

print("\nLoading Small (freq100) Graph...")
with open(SMALL_GRAPH_PATH, "rb") as f:
    G_small = pickle.load(f)
print(f"Small Graph loaded: {G_small.number_of_nodes():,} nodes, {G_small.number_of_edges():,} edges")

Loading Full Graph...
Full Graph loaded: 37,262 nodes, 1,533,339 edges

Loading Small (freq100) Graph...
Small Graph loaded: 16,279 nodes, 630,718 edges


In [23]:
def find_flights(G, origin, dest):
    """Finds all flights between origin and dest in the AoN graph."""
    # Remember: An AoN node is a tuple: (origin, dest, dep_time, arr_time)
    flights = [node for node in G.nodes() if node[0] == origin and node[1] == dest]
    # Sort by departure time for easy reading
    return sorted(flights, key=lambda x: x[2])

origin = "RIX"
dest = "VNO"

print(f"--- Flights from {origin} to {dest} in SMALL graph ---")
f_small = find_flights(G_small, origin, dest)
f_full = find_flights(G_full, origin, dest)
print(f_small==f_full)

--- Flights from RIX to VNO in SMALL graph ---
True


In [25]:
# Put the exact node tuple of your flight here
target_flight = ('BRU', 'CGN', 90, 137) # Example: BRU to CGN departing 01:30 (90 mins)

def check_connections(G, flight_node):
    if not G.has_node(flight_node):
        print(f"Flight {flight_node} does not exist in this graph.")
    else:
        print(f"Flight {flight_node} exists in this graph")

print("--- Connections in SMALL graph ---")
check_connections(G_small, target_flight)

print("\n--- Connections in FULL graph ---")
check_connections(G_full, target_flight)

--- Connections in SMALL graph ---
Flight ('BRU', 'CGN', 90, 137) does not exist in this graph.

--- Connections in FULL graph ---
Flight ('BRU', 'CGN', 90, 137) does not exist in this graph.


In [33]:
def check_route_validity(G, route_nodes, graph_name="Graph"):
    print(f"=== Tracing Route in {graph_name} ===")
    
    # Step 1: Check if the individual flights (nodes) exist at all
    nodes_exist = True
    for i, node in enumerate(route_nodes):
        if not G.has_node(node):
            print(f"❌ FATAL: The flight at Step {i} does not exist in the graph!")
            print(f"   Missing Flight: {node}")
            nodes_exist = False
    
    if not nodes_exist:
        print("\n🚨 Cannot check connections because some flights are missing entirely.")
        return
    else:
        print("✅ All individual flights exist in the graph. Checking connections...\n")

    # Step 2: Check if the layovers (edges) exist between the flights
    all_connections_valid = True
    for i in range(len(route_nodes) - 1):
        u = route_nodes[i]
        v = route_nodes[i+1]
        
        if G.has_edge(u, v):
            print(f"✅ Step {i} -> {i+1} connected (Layover at {u[1]})")
        else:
            print(f"❌ CONNECTION BROKEN between Step {i} and Step {i+1}")
            print(f"   Arriving on : {u}")
            print(f"   Departing on: {v}")
            
            # Quick diagnostic calculation
            layover_time = v[2] - u[3]
            print(f"   Diagnostic  : Layover duration would be {layover_time} minutes.")
            
            all_connections_valid = False
            break  # Stop checking at the first broken link
            
    if all_connections_valid:
        print("\n🎉 SUCCESS! The entire route is perfectly connected in this graph.")
    else:
        print("\n🚨 Route trace failed.")

# ==========================================
# PASTE YOUR ROUTE HERE
# Format: ('Origin', 'Dest', dep_time, arr_time)
# Note: Ensure the times exactly match what you see in Streamlit/live_checkpoint
# ==========================================
golden_route = [
    ('CDG', 'BRU', 1360, 1420),  # Flight: 60m 
    ('BRU', 'CGN', 1470, 1517),  # Wait: 50m, Flight: 47m
    ('CGN', 'MXP', 1565, 1655),  # Wait: 48m, Flight: 80m
    ('MXP', 'GVA', 1705, 1748),  # Wait: 60m, Flight: 43m
    ('GVA', 'CDG', 1770, 1845),  # Wait: 22m, Flight: 75m
    ('CDG', 'AMS', 1870, 1955),  # Wait: 25m, Flight: 85m
    ('AMS', 'OSL', 1985, 2090),  # Wait: 30m, Flight: 105m
    ('OSL', 'CPH', 2115, 2190),  # Wait: 25m, Flight: 75m
    ('CPH', 'GDN', 2220, 2280),  # Wait: 30m, Flight: 60m
    ('GDN', 'GOT', 2300, 2380),  # Wait: 20m, Flight: 40m
    ('GOT', 'HEL', 2435, 2520),  # Wait: 95m, Flight: 145m
    ('HEL', 'TLL', 2555, 2585),  # Wait: 35m, Flight: 30m
    ('TLL', 'RIX', 2605, 2655),  # Wait: 20m, Flight: 50m
    ('RIX', 'VNO', 2715, 2759)   # Wait: 60m, Flight: 44m
]

# Run the test on both graphs
check_route_validity(G_small, golden_route, graph_name="SMALL Graph (freq100)")
print("\n" + "="*50 + "\n")
check_route_validity(G_full, golden_route, graph_name="FULL Graph")

=== Tracing Route in SMALL Graph (freq100) ===
✅ All individual flights exist in the graph. Checking connections...

✅ Step 0 -> 1 connected (Layover at BRU)
✅ Step 1 -> 2 connected (Layover at CGN)
✅ Step 2 -> 3 connected (Layover at MXP)
✅ Step 3 -> 4 connected (Layover at GVA)
✅ Step 4 -> 5 connected (Layover at CDG)
✅ Step 5 -> 6 connected (Layover at AMS)
✅ Step 6 -> 7 connected (Layover at OSL)
✅ Step 7 -> 8 connected (Layover at CPH)
✅ Step 8 -> 9 connected (Layover at GDN)
✅ Step 9 -> 10 connected (Layover at GOT)
✅ Step 10 -> 11 connected (Layover at HEL)
✅ Step 11 -> 12 connected (Layover at TLL)
✅ Step 12 -> 13 connected (Layover at RIX)

🎉 SUCCESS! The entire route is perfectly connected in this graph.


=== Tracing Route in FULL Graph ===
✅ All individual flights exist in the graph. Checking connections...

✅ Step 0 -> 1 connected (Layover at BRU)
✅ Step 1 -> 2 connected (Layover at CGN)
✅ Step 2 -> 3 connected (Layover at MXP)
✅ Step 3 -> 4 connected (Layover at GVA)
✅ St

In [32]:

origin = "RIX"
dest = "VNO"

print(f"--- Flights from {origin} to {dest} in SMALL graph ---")
f_small = find_flights(G_small, origin, dest)
f_full = find_flights(G_full, origin, dest)
print(f_small)
print(f_full)

--- Flights from RIX to VNO in SMALL graph ---
[('RIX', 'VNO', 325, 375), ('RIX', 'VNO', 705, 755), ('RIX', 'VNO', 1075, 1125), ('RIX', 'VNO', 1275, 1319), ('RIX', 'VNO', 1765, 1815), ('RIX', 'VNO', 2145, 2195), ('RIX', 'VNO', 2515, 2565), ('RIX', 'VNO', 2715, 2759)]
[('RIX', 'VNO', 325, 375), ('RIX', 'VNO', 705, 755), ('RIX', 'VNO', 1075, 1125), ('RIX', 'VNO', 1275, 1319), ('RIX', 'VNO', 1765, 1815), ('RIX', 'VNO', 2145, 2195), ('RIX', 'VNO', 2515, 2565), ('RIX', 'VNO', 2715, 2759)]
